# Tutorial: Simulation-Based Inference for Astronomers

Welcome! This tutorial will guide you through the principles and application of Simulation-Based Inference (SBI), a powerful machine learning framework for performing parameter inference in scenarios where the likelihood function is intractable.

### The Scientific Problem

In many areas of astronomy, we have sophisticated forward models that can simulate complex phenomena—from the trajectory of a thrown object to the formation of the cosmic web. Given a set of physical parameters **`θ`**, our simulators can generate realistic data **`x`**.

However, the inverse problem, i.e. inferring the parameters **`θ`** that best explain some observed data **`x`**, is often incredibly difficult. Traditional methods like Markov Chain Monte Carlo (MCMC) require a "likelihood" function, $ P(x | \theta) $, which is often unknown or too computationally expensive to evaluate. This is where SBI  comes in. We will train a neural network to *learn* this inverse mapping directly from simulations. 

### Learning Objectives

By the end of this tutorial, you will be able to:
1.  Build a simple neural network to perform parameter regression.
2.  Upgrade this model to a **Neural Posterior Estimator (NPE)** to predict full, probabilistic posterior distributions.
3.  Construct a **Convolutional Neural Network (CNN)** to perform inference on 2D image data.
4.  Diagnose the calibration of a probabilistic model using P-P plots.
5.  Use **Optuna** to perform an automated search for optimal model hyperparameters.

### Notebook Outline
* **Section 1:** A Simple Neural Network for Parameter Inference
* **Section 2:** A Simple Neural Posterior Estimator (NPE)
* **Section 3:** A CNN for Inferring Cosmology from Dark Matter Maps
* **Hometask:** Optuna Hyperparameter Tuning

---

## Section 1: A Simple Neural Network for Parameter Inference

Our goal in this first section is to build a simple neural network that can learn to infer the parameters of a physical simulation.

**Our Plan:**
1.  **The Simulator:** We'll create a simulation of a ball being thrown. The simulation will take physical parameters (initial velocity, angle, drag) and produce noisy observations of the ball's trajectory.
2.  **The Dataset:** We'll use the simulator to generate thousands of examples to create training, validation, and test datasets.
3.  **The Inference Model:** We'll build and train a simple fully-connected neural network in PyTorch to recover the physical parameters from the observed trajectory.
4.  **Evaluation:** We'll evaluate our model's performance on the test set to see how well it learned the inverse problem.

In [ ]:
# !pip install numpy torch matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Setup device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### The Simulator ⚽

Our physical model simulates the 2D trajectory of a ball thrown in the air. The simulation is governed by three physical parameters, which we'll call **`theta`**:

1.  `v0`: The initial velocity of the ball (m/s).
2.  `alpha0`: The initial angle of the throw (degrees).
3.  `k`: A drag coefficient that slows the ball down.

The simulator's output, which we'll call the **data `x`**, will be the ball's (x, y) position at 10 fixed time snapshots. We also add a small amount of Gaussian noise to these positions to mimic measurement error.

The task of our neural network will be to take a data vector `x` (20 values) and predict the parameter vector `theta` (3 values).

In [ ]:
def simulator(theta, n_steps=10, t_max=2.0, noise_level=0.5):
    """
    Simulates the 2D trajectory of a thrown ball with drag.

    Args:
        theta (np.array): A vector of 3 parameters [v0, alpha0, k].
        n_steps (int): The number of time snapshots to record.
        t_max (float): The maximum simulation time.
        noise_level (float): Standard deviation of Gaussian noise to add.

    Returns:
        np.array: The flattened, noisy [x, y] positions at each time step.
    """
    v0, alpha0_deg, k = theta
    alpha0_rad = np.deg2rad(alpha0_deg)
    g = 9.81  # gravity

    # Initial conditions
    vx, vy = v0 * np.cos(alpha0_rad), v0 * np.sin(alpha0_rad)
    x, y = 0.0, 0.0
    dt = t_max / (n_steps * 10)  # Use finer steps for integration

    trajectory = []
    time_points = np.linspace(t_max / n_steps, t_max, n_steps)
    time_idx = 0

    for step in range(int(t_max / dt)):
        # Euler integration
        ax = -k * vx
        ay = -g - k * vy
        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt

        # Record position at specified time snapshots
        if time_idx < len(time_points) and (step * dt) >= time_points[time_idx]:
            trajectory.append([x, y])
            time_idx += 1
            if y < 0 and len(trajectory) > 1:  # Stop if ball hits ground
                break

    # Pad trajectory if simulation ended early
    while len(trajectory) < n_steps:
        trajectory.append(trajectory[-1])

    trajectory = np.array(trajectory)
    # Add noise and flatten
    noisy_trajectory = trajectory + \
        np.random.randn(*trajectory.shape) * noise_level
    return noisy_trajectory.flatten()

### Visualizing the Simulator

Before we generate a massive dataset, let's visualize a few sample trajectories. Notice how changing the parameters creates visibly different trajectories. This visible difference is what will allow our neural network to learn the mapping from trajectory to parameters.

In [ ]:
# Generate and plot a few examples
test_thetas = [
    [10.0, 30.0, 0.1],  # Low angle, low drag
    [20.0, 60.0, 0.1],  # High angle, low drag
    [20.0, 60.0, 0.5],  # High angle, high drag
]

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.axhline(0, color='k', linestyle='--')

for theta in test_thetas:
    # Simulate (flattened) data and reshape to (n_steps, 2)
    traj_flat = simulator(np.array(theta), noise_level=0.0)
    traj = traj_flat.reshape(-1, 2)

    ax.plot(traj[:, 0], traj[:, 1], 'o-',
            label=f"v0={theta[0]}, a0={theta[1]}, k={theta[2]}")

ax.set_xlabel("X Position (m)")
ax.set_ylabel("Y Position (m)")
ax.set_title("Sample Simulator Trajectories")
ax.legend()
ax.grid(True)
plt.show()

### Generating the Dataset

We'll use our simulator to generate a dataset of 50,000 examples. Each example consists of a parameter vector `theta` and the corresponding simulated data `x`. We'll then split this into training, validation, and test sets.

In [ ]:
# Define parameter priors (the range from which we'll draw random parameters)
param_priors = {
    'v0': (5.0, 25.0),      # m/s
    'alpha0': (20.0, 70.0),  # degrees
    'k': (0.1, 0.5)         # drag
}

n_samples = 50000
n_params = len(param_priors)
n_features = 20  # 10 (x,y) pairs

# Generate random parameters from a uniform distribution
thetas = np.zeros((n_samples, n_params))
for i, key in enumerate(param_priors):
    thetas[:, i] = np.random.uniform(*param_priors[key], size=n_samples)

# Run the simulator to generate the data
xs = np.array([simulator(theta)
              for theta in tqdm(thetas, desc="Generating data")])

# Convert to PyTorch tensors
X = torch.from_numpy(xs).float()
Y = torch.from_numpy(thetas).float()

In [ ]:
# Create datasets and dataloaders
batch_size = 128

# Split data: 70% train, 15% validation, 15% test
train_size = int(0.8 * n_samples)
val_size = int(0.1 * n_samples)
test_size = n_samples - train_size - val_size

dataset = TensorDataset(X, Y)
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Data shapes: X={X.shape}, Y={Y.shape}")
print(
    f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

### The Inference Model: A Fully-Connected Network

Now we'll define our inference model using PyTorch. A simple but effective choice is a Multi-Layer Perceptron (MLP), which consists of several fully-connected (`nn.Linear`) layers stacked together. We'll use the ReLU activation function between layers to introduce non-linearity.

**✏️ Exercise:** Complete the `nn.Sequential` block below to define the neural network. It should have:
1.  An input layer that matches our data's feature dimension (`n_features`).
2.  Three hidden layers with 64, 128, and 64 nodes, respectively. Use `nn.ReLU` as the activation function after each hidden layer.
3.  An output layer that matches the number of parameters we want to predict (`n_params`).

In [ ]:
# EXERCISE CELL
class SimpleNet(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.network = nn.Sequential(
            # --- YOUR CODE HERE ---

            # ----------------------
        )

    def forward(self, x):
        return self.network(x)

### Training the Model

To train the network, we need to define a **loss function** and an **optimizer**.
* **Loss Function:** Since this is a regression problem (predicting continuous values), the **Mean Squared Error (MSE)** is a standard choice. It measures the average squared difference between the true and predicted parameters.
* **Optimizer:** We'll use the **Adam optimizer**, a popular and effective algorithm for training deep neural networks.

The training process involves looping through the training data for a number of "epochs." In each step, we make predictions, calculate the loss, and use the optimizer to update the network's weights to minimize that loss.

In [ ]:
def mse_criterion(y_pred, y_batch):
    # Mean Squared Error (MSE) loss
    return torch.mean((y_pred - y_batch)**2)


def train_model(model, train_loader, val_loader, criterion, n_epochs=20, lr=1e-3):
    # Send the model to the configured device (GPU or CPU)
    model.to(device)
    # Initialize the Adam optimizer
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Dictionary to store loss history for plotting
    history = {'train_loss': [], 'val_loss': []}

    # Wrap the epoch loop with tqdm for a nice progress bar
    pbar = tqdm(range(n_epochs), desc="Training Epochs")
    for epoch in pbar:
        # --- Training Phase ---
        model.train()  # Set model to training mode (activates dropout, batchnorm, etc.)
        epoch_train_loss = 0
        # Loop over batches of data
        for x_batch, y_batch in train_loader:
            # Move data to the device
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            # 1. Zero out gradients from previous steps
            optimizer.zero_grad()

            # 2. Forward pass: compute predictions
            y_pred = model(x_batch)

            # 3. Compute the loss using the provided criterion function
            loss = criterion(y_pred, y_batch)

            # 4. Backward pass: compute gradients of the loss with respect to model parameters
            loss.backward()

            # 5. Optimizer step: update model weights
            optimizer.step()

            # Accumulate the loss for this epoch
            epoch_train_loss += loss.item()

        # --- Validation Phase ---
        model.eval()  # Set model to evaluation mode (disables dropout, etc.)
        epoch_val_loss = 0
        with torch.no_grad():  # Disable gradient calculation for efficiency
            # Loop over batches of validation data
            for x_batch, y_batch in val_loader:
                # Move data to the device
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                # 1. Forward pass
                y_pred = model(x_batch)

                # 2. Compute loss
                loss = criterion(y_pred, y_batch)

                # Accumulate validation loss
                epoch_val_loss += loss.item()

        # Calculate average losses for the epoch
        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_val_loss = epoch_val_loss / len(val_loader)
        # Store the average losses
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)

        # Update the tqdm progress bar description with the latest losses
        pbar.set_postfix({'train_loss': f"{avg_train_loss:.4f}",
                         'val_loss': f"{avg_val_loss:.4f}"})

    return history

In [ ]:
# Instantiate and train the model
model = SimpleNet(n_features, n_params)
history = train_model(model, train_loader, val_loader, 
                      criterion=mse_criterion, n_epochs=100, lr=1e-4)

# Plot training history
plt.figure(figsize=(8, 5))
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.title('Model Training History')
plt.show()

### Evaluating the Model

Now for the real test: how well does our trained model perform on data it has never seen before? We'll use our `test_loader` to make predictions and compare them to the true parameter values.

A good way to visualize this is a "true vs. predicted" scatter plot. If the model were perfect, all points would lie on the y=x line. We'll also look at the **residuals** (the difference between predicted and true values) to check for any systematic bias in our predictions. A good model should have residuals centered around zero.

In [ ]:
# Get predictions on the test set
model.eval()
y_true_list = []
y_pred_list = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_pred = model(x_batch)
        y_true_list.append(y_batch.cpu().numpy())
        y_pred_list.append(y_pred.cpu().numpy())

y_true = np.concatenate(y_true_list)
y_pred = np.concatenate(y_pred_list)

In [ ]:
# Plot true vs. predicted values
param_labels = [r'$v_0$', r'$\alpha_0$', r'$k$']
fig, axes = plt.subplots(1, n_params, figsize=(15, 5))

for i in range(n_params):
    min_val = min(y_true[:, i].min(), y_pred[:, i].min())
    max_val = max(y_true[:, i].max(), y_pred[:, i].max())

    axes[i].scatter(y_true[:, i], y_pred[:, i], alpha=0.2, s=10)
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')
    axes[i].set_xlabel(f'True {param_labels[i]}')
    axes[i].set_ylabel(f'Predicted {param_labels[i]}')
    axes[i].set_title(f'Parameter {i+1}')
    axes[i].grid(True)
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot histograms of the residuals
residuals = y_pred - y_true
fig, axes = plt.subplots(1, n_params, figsize=(15, 5))

for i in range(n_params):
    mean_resid = np.mean(residuals[:, i])
    std_resid = np.std(residuals[:, i])

    axes[i].hist(residuals[:, i], bins=50, density=True, alpha=0.7)
    axes[i].axvline(0, color='r', linestyle='--', label='Zero Bias')
    axes[i].set_xlabel(f'Residual (Predicted - True) for {param_labels[i]}')
    axes[i].set_ylabel('Density')
    axes[i].set_title(f"Mean={mean_resid:.3f}, Std={std_resid:.3f}")
    axes[i].grid(True)
    axes[i].legend()

plt.tight_layout()
plt.show()

### Additional tasks ✅
Consider trying the following further tasks to deepen your understanding:
- Experiment with different network architectures (more layers, different activation functions) to see how they affect performance.
- Test how the constraining power of the model changes with the amount of training data or the noise level in the observations.
- Add another parameter to the simulator (e.g., wind speed) and see if the model can still learn effectively.

### Section 1 Conclusion

Fantastic! We have successfully built and trained a neural network that can perform **amortized inference**. It takes noisy observational data and directly outputs an estimate of the underlying physical parameters.

The results look quite good—the true and predicted values are highly correlated, and the residuals are centered near zero. However, our model only gives us a single "point estimate" for each parameter. It doesn't tell us anything about the *uncertainty* in its prediction.

In the next section, we will upgrade our model to a **Neural Posterior Estimator (NPE)**, which will allow us to predict a full posterior probability distribution for each parameter, giving us both an estimate and its uncertainty.

---

## Section 2: A Simple Neural Posterior Estimator (NPE)

In the previous section, we built a model that provides a single "point estimate" for each parameter. While useful, a point estimate doesn't capture the **uncertainty** of the prediction. Is the model very confident or is it just a rough guess? To answer this, we need to predict a full probability distribution for each parameter. This is known as **posterior estimation**.

Our goal is to create a **Neural Posterior Estimator (NPE)**. We'll modify our network to output the parameters of a probability distribution (in this case, a Gaussian) for each physical parameter we want to infer.

**Our Plan:**
1.  **Modify the Network:** We'll change the output layer of our network to predict a **mean (`μ`)** and a **standard deviation (`σ`)** for each physical parameter.
2.  **A New Loss Function:** We'll replace the MSE with the **Negative Log-Likelihood (NLL)**. This will train the network to produce a distribution that is both centered on the true value and has an appropriate width.
3.  **Evaluation with Uncertainty:** We'll plot our predictions with error bars representing the predicted uncertainty.
4.  **Coverage Test:** We'll perform a crucial diagnostic test to see if our predicted uncertainties are statistically meaningful or "well-calibrated".

### Modifying the Network Architecture

To predict both a mean and a standard deviation for each of our `n_params` parameters, our network now needs to output `2 * n_params` values. We will interpret these outputs as follows:
* The first `n_params` outputs will be the **mean** vector, `μ`.
* The next `n_params` outputs will be the **log of the variance**, `log(σ²)`.

We predict the log-variance instead of the standard deviation directly for numerical stability and to ensure the predicted variance is always positive (since `σ² = exp(log(σ²))`).

**✏️ Exercise:** Copy your `SimpleNet` class from Section 1 into the cell below (we'll rename it `PosteriorNet`) and modify the final `nn.Linear` layer so that it has the correct number of output features.

In [ ]:
# EXERCISE CELL
class PosteriorNet(nn.Module):
    def __init__(self, in_features, out_params):
        super().__init__()
        # The number of output features should be twice the number of parameters
        out_features = 2 * out_params
        self.network = nn.Sequential(
            # --- COPY YOUR SimpleNet ARCHITECTURE HERE AND MODIFY THE LAST LAYER ---
            # --------------------------------------------------------------------
        )

    def forward(self, x):
        return self.network(x)

### The Loss Function: Negative Log-Likelihood

With our old MSE loss, the network was penalized only by the distance between the true and predicted values. Now, it must also learn to produce an appropriate uncertainty. The correct tool for this is the **Negative Log-Likelihood (NLL)**.

For a set of true parameters $ \theta $ and a network that predicts a Gaussian distribution for each parameter with mean $ \mu(x) $ and variance $ \sigma^2(x) $, the loss for a single data point is:

$$ \mathcal{L}(\theta) = -\log P(\theta | \mu(x), \sigma^2(x)) $$

For a Gaussian, this works out to:

$$ \mathcal{L}(\theta) = \frac{1}{2} \sum_{i=1}^{n_{params}} \left( \frac{(\theta_i - \mu_i)^2}{\sigma_i^2} + \log(\sigma_i^2) \right) + \text{const} $$

Since our network predicts `log_var` where `log_var = log(σ²)`, we can substitute `σ² = exp(log_var)` into the equation to get the final form we will implement.

**✏️ Exercise:** Complete the `nll_criterion` function below. It should take the network's raw output and the true parameter values, parse the output into `mean` and `log_var`, and then compute the loss.

In [ ]:
# EXERCISE CELL
def nll_criterion(y_pred, y_batch):
    """
    Computes the Gaussian Negative Log-Likelihood loss.

    Args:
        y_pred (torch.Tensor): Raw output from the network (shape: [batch_size, 2 * n_params]).
        y_batch (torch.Tensor): True parameter values (shape: [batch_size, n_params]).

    Returns:
        torch.Tensor: The mean loss for the batch.
    """
    # Get the number of parameters from the true values tensor
    n_params = y_batch.shape[1]

    # --- YOUR CODE HERE ---

    # 1. Split the network output into mean and log_var
    # Hint: Use tensor slicing, e.g., y_pred[:, :n_params]


    # 2. Calculate the NLL loss using the formula above
    # Hint: torch.exp() will be useful

    # ----------------------

    return torch.mean(loss)

### Training the NPE Model

Now we can train our new `PosteriorNet`. Since we made our `train_model` function more flexible, we can reuse it directly. All we need to do is pass our new `nll_criterion` as the `criterion` argument.

In [ ]:
# Instantiate and train the new model
npe_model = PosteriorNet(n_features, n_params)

# Call the original train_model function with the NLL criterion
npe_history = train_model(npe_model, train_loader, val_loader,
                          criterion=nll_criterion, n_epochs=100, lr=1e-4)

# Plot training history
plt.figure(figsize=(8, 5))
plt.plot(npe_history['train_loss'], label='Training Loss')
plt.plot(npe_history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('NLL Loss')
plt.legend()
plt.grid(True)
plt.title('NPE Model Training History')
plt.show()

### Evaluation with Uncertainty

Let's evaluate our NPE model on the test set. We'll make the same "true vs. predicted" plot as before, but this time we can add error bars to our predictions. The size of the error bar for each point will be the standard deviation (`σ`) predicted by the network for that specific data point.

In [ ]:
# Get predictions on the test set
npe_model.eval()
y_true_list = []
y_pred_list = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_pred = npe_model(x_batch)
        y_true_list.append(y_batch.cpu().numpy())
        y_pred_list.append(y_pred.cpu().numpy())

y_true = np.concatenate(y_true_list)
y_pred = np.concatenate(y_pred_list)

# Split predictions into mean and log_var, then calculate std
pred_mean = y_pred[:, :n_params]
pred_log_var = y_pred[:, n_params:]
pred_std = np.sqrt(np.exp(pred_log_var))

In [ ]:
# Plot true vs. predicted values with error bars
param_labels = [r'$v_0$', r'$\alpha_0$', r'$k$']
fig, axes = plt.subplots(1, n_params, figsize=(18, 6))

for i in range(n_params):
    min_val = y_true[:, i].min()
    max_val = y_true[:, i].max()

    # We plot a random subset of points for clarity
    subset = np.random.choice(len(y_true), size=500, replace=False)

    axes[i].errorbar(y_true[subset, i], pred_mean[subset, i], yerr=pred_std[subset, i],
                     fmt='o', alpha=0.3, markersize=5, label='Predicted Mean & Std')
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')
    axes[i].set_xlabel(f'True {param_labels[i]}')
    axes[i].set_ylabel(f'Predicted {param_labels[i]}')
    axes[i].set_title(f'Parameter {i+1}')
    axes[i].grid(True)
    axes[i].legend()

plt.tight_layout()
plt.show()

### Calibration Check: Posterior Coverage

The error bars look plausible, but are they statistically meaningful? A key test for any probabilistic model is to check its **calibration**.

Here's the idea: if our network predicts a 68% credible interval (which corresponds to $\mu \pm 1\sigma$ for a Gaussian), then we expect the true parameter value to fall within that interval 68% of the time. If it falls inside more often, our uncertainties are too large (overconfident). If it falls inside less often, our uncertainties are too small (underconfident). This test is known as checking the **posterior coverage**.

Let's calculate the empirical coverage for the 1-sigma (68.3%) interval for each parameter over our entire test set.

In [ ]:
# Calculate empirical coverage
one_sigma_coverage = 0.6827

for i in range(n_params):
    # Find the lower and upper bounds of the 1-sigma credible interval
    lower_bound = pred_mean[:, i] - pred_std[:, i]
    upper_bound = pred_mean[:, i] + pred_std[:, i]

    # Check how many true values fall within this interval
    is_covered = (y_true[:, i] >= lower_bound) & (y_true[:, i] <= upper_bound)

    # Calculate the fraction of covered points (the empirical coverage)
    empirical_coverage = np.mean(is_covered)

    print(f"Parameter: {param_labels[i]}")
    print(f"  - Expected Coverage (1-sigma): {one_sigma_coverage:.3f}")
    print(f"  - Empirical Coverage:          {empirical_coverage:.3f}")
    print("-" * 20)

### Additional tasks ✅
Consider trying the following further tasks to deepen your understanding:
- Add a more complex predictive distribution, like a mixture of Gaussians or a Gaussian with a learned correlation structure between parameters. This will require modifying the loss function.
- Change the noise level in the test observations, and see how the neural network generalizes to data with different noise characteristics.
- Add learning rate scheduling and early stopping to improve training stability and prevent overfitting.

### Section 2 Conclusion

Amazing! Our NPE model not only provides accurate point estimates (the means), but it also provides uncertainty estimates (`σ`) that are well-calibrated. The empirical coverage is very close to the expected 68.3%, indicating that the model has learned a meaningful posterior distribution.

We have now built a powerful tool for amortized posterior estimation on simple vector data. In the next section, we'll scale this up to a much more challenging and realistic scientific problem: inferring physical parameters from 2D images using a **Convolutional Neural Network (CNN)**.

---

## Section 3: A CNN for Inferring Cosmology from Dark Matter Maps

In the previous sections, we built a probabilistic inference model for a simple toy problem. Now, we'll apply this powerful technique to a frontier problem in cosmology: inferring the fundamental parameters of our Universe from maps of its large-scale structure.

The cosmic web, a vast network of dark matter filaments and halos, is shaped by the underlying cosmology. Different values for parameters like the matter density ($\Omega_m$) or the amplitude of fluctuations ($\sigma_8$) produce statistically different structures. We will train a **Convolutional Neural Network (CNN)** to learn this mapping and infer cosmological parameters directly from 2D dark matter density maps.

**Our Plan:**
1.  **The Dataset:** We will load a dataset of 2D dark matter maps, representative of slices from Lagrangian Perturbation Theory (LPT) simulations, and their corresponding cosmological parameters.
2.  **The Model:** We'll use the same hybrid CNN-NPE architecture. The CNN will learn to extract summary statistics from the maps, and the dense head will estimate the posterior distribution of the cosmological parameters.
3.  **GPU Training:** Training on images is computationally expensive, so we'll leverage a GPU if available.
4.  **Evaluation:** We'll perform the same evaluation as before, checking if our network can produce accurate and well-calibrated constraints on cosmology.

### The Image Dataset

Our dataset consists of:
* **`x`**: A `64x64` pixel 2D map representing the dark matter overdensity field, $ \delta = (\rho - \bar{\rho})/\bar{\rho} $. [Image of a dark matter simulation slice]
* **`theta`**: A vector of 5 cosmological parameters that govern the evolution of the Universe:
    * `Ωₘ (Omega_m)`: The total matter density parameter.
    * `Ωb (Omega_b)`: The baryon (normal matter) density parameter.
    * `h`: The Hubble constant, scaled by 100 km/s/Mpc.
    * `nₛ (n_s)`: The scalar spectral index of primordial fluctuations.
    * `σ₈ (sigma_8)`: The amplitude of matter fluctuations on 8 Mpc/h scales.

As before, the notebook will load data from local `x.npy` and `theta.npy` files if they exist. If not, it will run a **toy simulator** to generate them.

**Note:** The following function is a *toy model*. It is not a full LPT simulation. It generates images with statistical properties (e.g., number and clumpiness of halos) that are correlated with the input cosmology, which is sufficient for this tutorial's purpose.

In [ ]:
import os


def generate_dm_maps(n_samples=4000, img_size=64):
    print(f"Generating {n_samples} new dark matter maps (toy model)...")
    # Plausible priors for cosmological parameters
    priors = {
        'Omega_m': (0.1, 0.5),
        'Omega_b': (0.03, 0.07),
        'h': (0.5, 0.9),
        'n_s': (0.9, 1.1),
        'sigma_8': (0.6, 1.0),
    }
    n_params = len(priors)
    thetas = np.zeros((n_samples, n_params))
    for i, key in enumerate(priors):
        thetas[:, i] = np.random.uniform(*priors[key], size=n_samples)

    xs = np.zeros((n_samples, 1, img_size, img_size))

    for i in tqdm(range(n_samples), desc="Creating maps"):
        Omega_m, _, _, _, sigma_8 = thetas[i]

        # Toy model: Number of halos depends on Omega_m, their peakiness on sigma_8
        n_halos = int(5 + Omega_m * 40)
        halo_amplitude = 1.0 + sigma_8 * 2.0

        # Create a base of Gaussian random noise
        field = np.random.randn(img_size, img_size) * 0.1

        # Add 'halos' (Gaussian blobs)
        for _ in range(n_halos):
            x0, y0 = np.random.randint(0, img_size, 2)
            sx, sy = np.random.uniform(2, 5, 2)
            A = np.random.uniform(0.5, 1.0) * halo_amplitude

            coords_x, coords_y = np.meshgrid(
                np.arange(img_size), np.arange(img_size))
            blob = A * np.exp(-(((coords_x-x0)**2 / (2*sx**2)
                                 ) + ((coords_y-y0)**2 / (2*sy**2))))
            field += blob

        # Smooth the field to create filament-like structures
        from scipy.ndimage import gaussian_filter
        field = gaussian_filter(field, sigma=1.2)

        xs[i, 0, :, :] = field

    return xs, thetas


# --- Main Data Loading ---
x_path = 'x.npy'
theta_path = 'theta.npy'

if os.path.exists(x_path) and os.path.exists(theta_path):
    print("Loading existing data from disk...")
    xs_img = np.load(x_path)
    thetas_img = np.load(theta_path)
else:
    xs_img, thetas_img = generate_dm_maps()
    # np.save(x_path, xs_img)
    # np.save(theta_path, thetas_img)

print(f"Data shapes: x={xs_img.shape}, theta={thetas_img.shape}")

In [ ]:
# Visualize a few examples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(xs_img[i, 0], cmap='inferno')
    ax.set_title(f"Om={thetas_img[i, 0]:.2f}, s8={thetas_img[i, 4]:.2f}")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create datasets and dataloaders for the image data
X_img = torch.from_numpy(xs_img).float()
Y_img = torch.from_numpy(thetas_img).float()

n_samples_img = X_img.shape[0]
n_params_img = Y_img.shape[1]

# Re-split data
train_size = int(0.7 * n_samples_img)
val_size = int(0.15 * n_samples_img)
test_size = n_samples_img - train_size - val_size

img_dataset = TensorDataset(X_img, Y_img)
train_img_dataset, val_img_dataset, test_img_dataset = torch.utils.data.random_split(
    img_dataset, [train_size, val_size, test_size])

train_img_loader = DataLoader(train_img_dataset, batch_size=64, shuffle=True)
val_img_loader = DataLoader(val_img_dataset, batch_size=64, shuffle=False)
test_img_loader = DataLoader(test_img_dataset, batch_size=64, shuffle=False)

### The Hybrid CNN-NPE Model

The model architecture consists of a **CNN Embedder** to extract features from the 2D map, and a **Dense Head** to perform the posterior estimation based on those features.

**✏️ Exercise:** Create three blocks, each with a `Conv2d`, `ReLU`, and `MaxPool2d` layer.
* Use a kernel size of 3 and padding of 1 for all `Conv2d` layers.
* Increase the number of channels: `1 -> 16`, `16 -> 32`, `32 -> 64`.
* Use a kernel size of 2 for all `MaxPool2d` layers.

In [ ]:
# EXERCISE CELL
class CNN_Embedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = ...

    def forward(self, x):
        return self.network(x)


# Test the embedder to find its output shape
embedder = CNN_Embedder()
dummy_img = torch.randn(1, 1, 64, 64)
embedding = embedder(dummy_img)
cnn_out_features = np.prod(embedding.shape[1:])
print(f"Size of flattened feature vector: {cnn_out_features}")

In [ ]:
# Assembling the full model
class CNN_NPE(nn.Module):
    def __init__(self, cnn_out_features, n_params):
        super().__init__()
        self.embedder = CNN_Embedder()
        self.dense_head = nn.Sequential(
            nn.Linear(cnn_out_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_params * 2)  # *2 for mean and log_var
        )

    def forward(self, x):
        embedding = self.embedder(x)
        embedding_flat = embedding.view(x.shape[0], -1)
        out = self.dense_head(embedding_flat)
        return out

### Training the CNN-NPE on Cosmological Data

We'll now train the model. The process is identical: we use the `train_model` function with the `nll_criterion`. Training will be slow without a GPU; 20 epochs is a good starting point to see if it's learning.

In [ ]:
# Instantiate the full CNN-NPE model
cnn_npe_model = CNN_NPE(cnn_out_features, n_params_img)

# Train the model
cnn_npe_history = train_model(
    cnn_npe_model,
    train_img_loader,
    val_img_loader,
    criterion=nll_criterion,
    n_epochs=20,  # Increase if you have time/GPU
    lr=1e-4
)

In [ ]:
# Plot training history
plt.figure(figsize=(8, 5))
plt.plot(cnn_npe_history['train_loss'], label='Training Loss')
plt.plot(cnn_npe_history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('NLL Loss')
plt.legend()
plt.grid(True)
plt.title('CNN-NPE Model Training History')
plt.show()

### Evaluating the Cosmology-Inference Network

Let's see how well our network learned to constrain cosmology. We perform the same two checks: the true vs. predicted plot with error bars, and the posterior coverage test.

In [ ]:
# Get predictions on the test set
cnn_npe_model.eval()
y_true_list_img = []
y_pred_list_img = []

with torch.no_grad():
    for x_batch, y_batch in test_img_loader:
        x_batch = x_batch.to(device)
        y_pred = cnn_npe_model(x_batch)
        y_true_list_img.append(y_batch.cpu().numpy())
        y_pred_list_img.append(y_pred.cpu().numpy())

y_true_img = np.concatenate(y_true_list_img)
y_pred_img = np.concatenate(y_pred_list_img)

# Split predictions into mean and std
pred_mean_img = y_pred_img[:, :n_params_img]
pred_log_var_img = y_pred_img[:, n_params_img:]
pred_std_img = np.sqrt(np.exp(pred_log_var_img))

In [ ]:
# Plot true vs. predicted values with error bars
param_labels_img = [r'$\Omega_m$', r'$\Omega_b$',
                    r'$h$', r'$n_s$', r'$\sigma_8$']
fig, axes = plt.subplots(1, n_params_img, figsize=(22, 4))

for i in range(n_params_img):
    min_val = y_true_img[:, i].min()
    max_val = y_true_img[:, i].max()

    axes[i].errorbar(y_true_img[:, i], pred_mean_img[:, i], yerr=pred_std_img[:, i],
                     fmt='o', alpha=0.3, markersize=4)
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')
    axes[i].set_xlabel(f'True {param_labels_img[i]}')
    axes[i].set_ylabel(f'Predicted {param_labels_img[i]}')
    axes[i].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate empirical coverage for the CNN-NPE model
one_sigma_coverage = 0.6827
print("--- CNN-NPE Coverage Test ---")
for i in range(n_params_img):
    lower_bound = pred_mean_img[:, i] - pred_std_img[:, i]
    upper_bound = pred_mean_img[:, i] + pred_std_img[:, i]
    is_covered = (y_true_img[:, i] >= lower_bound) & (
        y_true_img[:, i] <= upper_bound)
    empirical_coverage = np.mean(is_covered)

    print(f"Parameter: {param_labels_img[i]}")
    print(f"  - Expected Coverage: {one_sigma_coverage:.3f}")
    print(f"  - Empirical Coverage: {empirical_coverage:.3f}")

In [ ]:
# Create P-P plots to check posterior calibration
from scipy.stats import norm

one_sigma_coverage = 0.6827
param_labels_img = [r'$\Omega_m$', r'$\Omega_b$',
                    r'$h$', r'$n_s$', r'$\sigma_8$']
fig, axes = plt.subplots(1, n_params_img, figsize=(22, 4.5))
fig.suptitle(
    'Calibration Check: Probability-Probability (P-P) Plots', fontsize=16)

# Define the expected quantiles (the x-axis)
expected_quantiles = np.linspace(0, 1, 101)

for i in range(n_params_img):
    # For each test sample, find the percentile of the true value within its predicted Gaussian posterior
    # This is done using the Cumulative Distribution Function (CDF)
    percentiles = norm.cdf(
        y_true_img[:, i], loc=pred_mean_img[:, i], scale=pred_std_img[:, i])

    # Calculate the empirical quantiles (the y-axis)
    # For each expected quantile, what fraction of true values actually fell below it?
    empirical_quantiles = [np.mean(percentiles <= q)
                           for q in expected_quantiles]

    # Plot the results
    axes[i].plot(expected_quantiles, empirical_quantiles,
                 'b-', lw=2, label='Model Calibration')
    axes[i].plot([0, 1], [0, 1], 'r--', label='Ideal Calibration')
    axes[i].set_xlabel('Expected Cumulative Probability')
    axes[i].set_ylabel('Empirical Cumulative Probability')
    axes[i].set_title(param_labels_img[i])
    axes[i].grid(True)
    axes[i].legend()
    axes[i].set_aspect('equal')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### How to Interpret P-P Plots

These P-P plots are a powerful tool for diagnosing the calibration of our probabilistic model. Here's how to read them:

* **The Ideal Line (Red, Dashed):** The y=x line represents perfect calibration. If the model's predictions were perfectly calibrated, the blue curve would lie exactly on top of this line.
* **The Model Curve (Blue):** This curve shows the model's actual performance. The x-axis shows a percentile (e.g., 0.4 for the 40th percentile), and the y-axis shows what fraction of the true parameter values actually fell below that percentile in their respective predicted distributions.
* **Interpreting Deviations:**
    * If the blue curve is **above** the red line, the model's predicted uncertainties are generally too **small** (it is overconfident). For example, it might think only 30% of true values should be below a certain point, but in reality 40% are.
    * If the blue curve is **below** the red line, the model's predicted uncertainties are generally too **large** (it is underconfident).

A well-calibrated model will have a blue curve that closely follows the red dashed line, as we see here.

### Additional tasks ✅
Consider trying the following further tasks to deepen your understanding:
- Try modifying the CNN architecture or training hyperparameters to see how they affect performance and calibration.
- Try changing the resolution of the input images (e.g., 32x32 or 128x128) and see how it impacts the constraints on cosmology. (Hint: The scatter in the predictions should scale as $N_{\rm pixels}^{-1/2}$.)
- Implement data augmentation techniques (e.g., random rotations, flips) during training to improve model robustness.

### Section 3 Conclusion

We've built a model that can look at a map of dark matter and produce calibrated constraints on the fundamental parameters of our Universe. This process, which once required complex, hand-crafted summary statistics, can now be learned automatically by a neural network.

The true power of this "amortized" approach is its speed: once trained, the network can perform inference on a new map in milliseconds, a task that would be computationally prohibitive with traditional methods.

For students who have finished early, the final section provides an introduction to **hyperparameter optimization** using Optuna, a tool to automatically find the best settings for models like this one.

## Tutorial Conclusion

Congratulations on completing this tutorial! 🎉

We have journeyed from a very simple concept—a network to infer the parameters of a thrown ball—all the way to a sophisticated, automatically tuned inference engine for constraining cosmology from dark matter maps.

**Key Takeaways:**
* **Simulation-Based Inference (SBI)** is a powerful technique for solving inverse problems when you have a good forward simulator.
* **Neural Posterior Estimation (NPE)** allows us to move beyond simple point estimates to predict full, calibrated posterior distributions, giving us meaningful uncertainties.
* **Convolutional Neural Networks (CNNs)** are the essential tool for applying these techniques to complex scientific image data.

You now have a powerful set of tools and concepts to apply to your own research problems. Good luck!

---

## Hometask: Optuna Hyperparameter Tuning

We've successfully built and trained a sophisticated inference network. But how did we choose its architecture? How many layers should it have? How many nodes per layer? What is the optimal learning rate? These choices, known as **hyperparameters**, can have a huge impact on model performance.

Manually testing different combinations is tedious and inefficient. In this section, we'll use **Optuna**, a modern optimization framework, to automatically search for the best hyperparameters for our CNN-NPE model.

**Our Plan:** 
1.  **The Objective Function:** We'll wrap our model-building and training process into a single "objective" function that Optuna can call.
2.  **Defining the Search Space:** Inside this function, we'll define a search space for Optuna to explore for hyperparameters like learning rate, number of layers, etc.
3.  **Running the Study:** We'll launch an Optuna "study" to run multiple training trials and intelligently find the best-performing combination.
4.  **Analyzing Results:** We'll use Optuna's powerful built-in tools to visualize the search and identify the most important hyperparameters.

In [ ]:
# !pip install optuna plotly

In [ ]:
import optuna
import importlib

importlib.reload(optuna)

### The Objective Function

The core of any Optuna search is the **objective function**. This is a function that you write, which Optuna calls repeatedly. For each call (a "trial"), Optuna provides a set of suggested hyperparameters. Your function must use these hyperparameters to build, train, and evaluate a model, then return a single number (a metric) that Optuna should minimize or maximize.

In our case, the objective function will build a `CNN_NPE` model, train it for a few epochs on the dark matter map dataset, and return the final **validation loss**. Optuna's goal will be to find the set of hyperparameters that results in the lowest validation loss.

**✏️ Exercise:** Complete the `objective` function below. Your task is to use the `trial` object to define the search space. Use the following methods:
* `trial.suggest_float("lr", 1e-5, 1e-3, log=True)` for the learning rate.
* `trial.suggest_int("n_layers", 1, 3)` for the number of dense layers in the head.
* `trial.suggest_int("n_units_l0", 32, 256)` for the number of units in the first dense layer.
* `trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])` for the choice of optimizer.

In [ ]:
# EXERCISE CELL
def objective(trial):
    ...

### Running the Study

Now that we have our objective function, we can create an Optuna `study` and run the optimization. A "study" is the entire optimization session. We'll tell Optuna we want to `minimize` our objective and run it for `10 trials`.

**Note:** A real hyperparameter search would use hundreds or thousands of trials. We're using a small number here so it completes in a reasonable amount of time.

In [ ]:
# Create a study object and specify the direction is "minimize"
study = optuna.create_study(direction="minimize")

# Start the optimization
study.optimize(objective, n_trials=10)

print("Study finished!")

### Analyzing the Results

After the study is complete, Optuna makes it easy to inspect the results. We can find the best value, the best set of hyperparameters, and visualize the search process.

In [ ]:
# Print the results
print(f"Best trial final value (validation loss): {study.best_value}")
print("Best hyperparameters found:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

In [ ]:
# Optuna's built-in plotting functions are excellent
# Plot 1: Optimization history
optuna.visualization.plot_optimization_history(study)

In [ ]:
# Plot 2: Hyperparameter importances
# This shows which hyperparameters had the biggest impact on the final loss
optuna.visualization.plot_param_importances(study)

In [ ]:
# Plot 3: Slice plot
# This shows how the loss varies as a function of each hyperparameter
optuna.visualization.plot_slice(study)

### Additional tasks ✅
Consider trying the following further tasks to deepen your understanding:
- Train a model with the best hyperparameters found by Optuna and evaluate its performance on the test set.
- Experiment with adding more hyperparameters to the search space, such as the design of the CNN architecture (number of filters, kernel sizes) or training parameters (batch size, weight decay).
- Consider adding more objectives, like minimizing the average calibration bias in the predictions.